[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/03_sentiment_im_grossen.ipynb)

# Sitzung 3 — Sentiment im Großen

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI**

Letzte Woche: *eine* Bewertung einordnen. Heute: *hunderte* auf einmal — und die ersten Kennzahlen. Hier fängt „Analytics“ an.

> 💡 Wie immer: alles läuft über den **Mock**, key-frei. Der echte Aufruf kommt nur auf einer kleinen Stichprobe vor (Abschnitt 5).

## 0. Setup — einmal ausführen

In [ ]:
!pip install anthropic --quiet
import json, re, random
from collections import Counter
from datetime import date, timedelta
print('Fertig.')

## 1. Woher kommen hunderte Bewertungen?

Wir **erzeugen** unseren eigenen Datensatz — synthetische Bewertungen zu einem *fiktiven* Produkt (Nimbus Q2). Warum synthetisch? Keine Lizenzprobleme, und wir können das Chaos echter Daten gezielt einbauen (Sarkasmus, Fakes, gemischte Meinungen). Mehr dazu in einer späteren Sitzung.

Führt die Zelle aus — sie definiert den Generator. Details egal; wichtig ist die Funktion **`generate(n)`**.

In [ ]:
"""
Synthetic review corpus generator for the BDA course.

Fictional product: the "Nimbus Q2" - a wireless noise-cancelling earbud set
from a made-up brand ("Nimbus Audio"). Fully fictional so there is no real
brand, no real person, and no licensing question - we own this data outright.

The generator deliberately seeds the MESSINESS the course teaches against.
Ground-truth labels ARE recorded (true_sentiment, is_sarcastic, is_fake)
for later gold-set / evaluation sessions.
"""

import random
import json
import csv
from datetime import date, timedelta

SEED = 42
PRODUCT = "Nimbus Q2"
BRAND = "Nimbus Audio"

POS_ASPECTS = [
    ("Der Klang ist wirklich hervorragend, satte Bässe", "positive"),
    ("Die Geräuschunterdrückung funktioniert im Zug erstaunlich gut", "positive"),
    ("Akku hält locker den ganzen Arbeitstag", "positive"),
    ("Sitzt bequem im Ohr, auch nach Stunden", "positive"),
    ("Verbindung per Bluetooth ist sofort da und stabil", "positive"),
    ("Für den Preis absolut top verarbeitet", "positive"),
]
NEG_ASPECTS = [
    ("Die App stürzt ständig ab", "negative"),
    ("Nach zwei Wochen hat der rechte Ohrhörer aufgehört zu laden", "negative"),
    ("Die Geräuschunterdrückung rauscht hörbar", "negative"),
    ("Viel zu teuer für das, was man bekommt", "negative"),
    ("Die Touch-Steuerung reagiert kaum", "negative"),
    ("Das Case fühlt sich billig und klapprig an", "negative"),
]
NEUTRAL = [
    ("Ganz okay, nichts Besonderes, erfüllt seinen Zweck", "neutral"),
    ("Habe sie seit gestern, kann noch nicht viel sagen", "neutral"),
    ("Standard-Ohrhörer, wie erwartet", "neutral"),
]
SARCASTIC = [
    "Super, schon nach drei Tagen kaputt. Echt klasse Qualität.",
    "Toll, dass die App jedes Mal abstürzt. Genau das wollte ich.",
    "Wunderbar, 200 Euro für Ohrhörer, die nach links driften. Ein Traum.",
    "Ganz großes Kino, der Akku hält sagenhafte 40 Minuten.",
]
FAKE = [
    "BESTES PRODUKT EVER!!! Kauft alle bei www.super-deals-guenstig.example!!!",
    "5 Sterne 5 Sterne 5 Sterne bester shop schnelle lieferung AAA+++",
    "Gratis Gutschein Code NIMBUS100 auf meiner Seite klickt hier jetzt!!!",
    "amazing product best quality buy now discount link in profile",
]
ENGLISH = [
    ("Sound quality is great but the app is a disaster.", "mixed"),
    ("Battery life is amazing, best earbuds I've owned.", "positive"),
    ("Stopped working after a week, very disappointed.", "negative"),
]
JUNK = ["", "   ", ".", "???", "kein kommentar"]

def _mixed(rng):
    pos, _ = rng.choice(POS_ASPECTS)
    neg, _ = rng.choice(NEG_ASPECTS)
    connector = rng.choice([" - aber ", ", allerdings ", ". Leider "])
    return pos + connector + neg[0].lower() + neg[1:], "mixed"

def generate(n=300, seed=SEED):
    rng = random.Random(seed)
    start = date(2026, 1, 1)
    rows = []
    for i in range(n):
        r = rng.random()
        if r < 0.34:
            text, truth = rng.choice(POS_ASPECTS)
        elif r < 0.60:
            text, truth = rng.choice(NEG_ASPECTS)
        elif r < 0.72:
            text, truth = _mixed(rng)
        elif r < 0.80:
            text, truth = rng.choice(NEUTRAL)
        elif r < 0.88:
            text, truth = rng.choice(SARCASTIC), "negative"
        elif r < 0.94:
            text, truth = rng.choice(FAKE), "fake"
        elif r < 0.98:
            text, truth = rng.choice(ENGLISH)
        else:
            text, truth = rng.choice(JUNK), "junk"
        is_sarcastic = text in SARCASTIC
        is_fake = text in FAKE
        day_offset = int(rng.triangular(0, 270, 200 if truth == "negative" else 90))
        rating = _rating_for(truth, rng)
        rows.append({
            "review_id": f"R{i:04d}",
            "date": (start + timedelta(days=day_offset)).isoformat(),
            "product": PRODUCT,
            "rating": rating,
            "text": text,
            "true_sentiment": truth,
            "is_sarcastic": is_sarcastic,
            "is_fake": is_fake,
        })
    if n > 20:
        rows.append(dict(rows[5], review_id=f"R{n:04d}"))
        rows.append(dict(rows[12], review_id=f"R{n+1:04d}"))
    rng.shuffle(rows)
    return rows

def _rating_for(truth, rng):
    if truth == "positive": return rng.choice([4, 5, 5])
    if truth == "negative": return rng.choice([1, 1, 2])
    if truth == "mixed": return rng.choice([2, 3, 4])
    if truth == "neutral": return 3
    if truth == "fake": return 5
    return rng.choice([1, 3, 5])

Und die `classify`-Funktion von letzter Woche (Mock + echt):

In [ ]:
import json, re

def _mock_classify(text):
    t = (text or "").lower()
    pos = sum(w in t for w in ["gut","top","super","toll","hervorragend","bequem",
                                "stabil","klasse","liebe","genial","perfekt","great","love"])
    neg = sum(w in t for w in ["schlecht","kaputt","teuer","stürzt","rauscht","billig",
                                "nervt","enttäuscht","langsam","mangel","bad","broken"])
    if pos and neg: s = "mixed"
    elif pos: s = "positive"
    elif neg: s = "negative"
    else: s = "neutral"
    return {"sentiment": s, "reason": f"Mock: {pos} pos-, {neg} neg-Signale."}

def _real_classify(text, client, model="claude-sonnet-5"):
    prompt = (f'Ordne die folgende Produktbewertung ein. Antworte NUR mit JSON:\n'
              f'{{"sentiment": "positive|negative|neutral|mixed", "reason": "kurze Begruendung"}}\n\n'
              f'Bewertung: """{text}"""')
    resp = client.messages.create(model=model, max_tokens=200,
                                  messages=[{"role":"user","content":prompt}])
    raw = next((b.text for b in resp.content if hasattr(b,"text")), "")
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    return json.loads(m.group(0)) if m else {"sentiment":"neutral","reason":"parse error"}

def classify(text, use_real=False, client=None):
    return _real_classify(text, client) if use_real else _mock_classify(text)

Jetzt erzeugen wir 300 Bewertungen und schauen in die ersten hinein:

In [ ]:
reviews = generate(300)
print(f'{len(reviews)} Bewertungen erzeugt.\n')
for r in reviews[:5]:
    print(f"[{r['rating']}*] {r['text'][:60]}")

## 2. Von einer zu hunderten: die Schleife

In Sitzung 2 haben wir *eine* Bewertung eingeordnet. Um *alle* einzuordnen, wiederholen wir das in einer **Schleife** — für jede Bewertung ein Aufruf.

> 💡 Beim Mock geht das in einer Sekunde. Beim *echten* LLM wäre das hunderte Aufrufe — langsam und kostenpflichtig. Merkt euch das (Thema in Sitzung 11).

In [ ]:
analysiert = []
for r in reviews:
    ergebnis = classify(r['text'])          # Mock
    analysiert.append({**r, **ergebnis})
print('Analysiert:', len(analysiert))
print('Beispiel  :', analysiert[0]['sentiment'], '|', analysiert[0]['text'][:50])

## 3. Aggregieren: das Gesamtbild

Eine einzelne Bewertung ist eine Anekdote. **Hunderte zusammen** sind ein Signal. Zählen wir, wie oft jedes Sentiment vorkommt:

In [ ]:
counts = Counter(a['sentiment'] for a in analysiert)
print(counts)

n = len(analysiert)
print('\nVerteilung:')
for s, c in counts.most_common():
    print(f'  {s:9} {c:4}  ({round(100*c/n)}%)')

Das ist eure erste **Kennzahl**: der Anteil positiver / negativer Bewertungen über den ganzen Datensatz. Genau das will eine Produktmanagerin auf einen Blick.

## 4. Eure Aufgabe: vergleichen

Eine Kennzahl über *alles* ist nett. Interessanter wird es beim **Vergleichen**. Nehmt eine **Teilmenge** (z. B. nur schlecht bewertete) und schaut, wie sich das Sentiment unterscheidet.

In [ ]:
# Teilmenge: nur Bewertungen mit 1 oder 2 Sternen
schlecht = [a for a in analysiert if a['rating'] <= 2]
print(f'{len(schlecht)} Bewertungen mit <=2 Sternen')
print(Counter(a['sentiment'] for a in schlecht))

> ✏️ **Eure Aufgabe:** Ändert die Bedingung. Vergleicht z. B. `rating >= 4` (gute Sterne) mit dem Gesamtbild. Stimmen Sterne und LLM-Sentiment überein? Wo weichen sie ab — und warum könnte das sein?

## Der tiefere Punkt: braucht man *alle* Daten?

Wir haben 300 Bewertungen eingeordnet. Aber: braucht man wirklich *alle*, um das Gesamtbild zu kennen? Oft reicht eine **Stichprobe** — eine zufällige Auswahl, die den Datensatz repräsentiert. Das spart Zeit und (beim echten LLM) Geld.

In [ ]:
stichprobe = random.sample(analysiert, 50)   # 50 zufällig gezogen
c_gesamt = Counter(a['sentiment'] for a in analysiert)
c_probe  = Counter(a['sentiment'] for a in stichprobe)
print('Gesamt (300):', {k: round(100*v/300) for k,v in c_gesamt.items()})
print('Probe  (50) :', {k: round(100*v/50)  for k,v in c_probe.items()})

> 💡 **Kernpunkt:** Die Stichprobe kommt dem Gesamtbild oft nah — aber nicht exakt. Wie groß eine Stichprobe sein muss, um verlässlich zu sein, ist eine echte statistische Frage (klassische Big-Data-Analytics!). Für jetzt: weniger Daten können reichen, wenn sie *repräsentativ* sind.

## 5. Kurz echt: eine kleine Stichprobe mit dem echten LLM

Alles bisher lief über den Mock. Zum Abschluss: dieselbe Schleife, aber echt — und **nur auf 5 Bewertungen** (echte Aufrufe kosten Zeit und Geld).

> ℹ️ Nur mit Schlüssel (Secrets-Panel, `ANTHROPIC_API_KEY`). Ohne Schlüssel überspringen — der Mock reicht.

In [ ]:
# Nur mit Schlüssel:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

for r in reviews[:5]:
    e = classify(r['text'], use_real=True, client=client)
    print(f"{e['sentiment']:9} | {r['text'][:55]}")

> 🎓 **Vergleich:** Wie ordnet das echte LLM diese 5 ein — verglichen mit dem Mock? Achtet besonders auf die sarkastischen und gemischten Fälle.

## 6. Geschafft — und Ausblick

Ihr habt hunderte Bewertungen eingeordnet und zu Kennzahlen verdichtet — das ist der Kern von *Analytics*.

**Nächste Woche (28.10):** die Daten sind messy (fehlende Felder, Duplikate, Fremdsprachen). Wir bringen sie in Form — bevor wir in Sitzung 5 anfangen, *Themen* zu extrahieren.

> 💡 Von *einer* Zahl (%) zu *vielen* Einsichten: das bauen wir Schritt für Schritt aus.